# R3-01 · Introducción al NLP — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional y de **transferencia**: el mismo pipeline (TF-IDF + regresión logística), con un texto
muy distinto — el **producto** de 20.000 reclamos SERNAC de 2011 (dato histórico; SERNAC no
publica una base abierta más reciente) — para predecir el **mercado**. Texto corto y muchos
datos dan una exactitud altísima… que esconde a las clases pequeñas. Verás por qué el problema
no es el modelo, sino lo que el texto **no dice**.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, urllib.request
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), se baja del repo.
if not os.path.exists("reclamos.csv"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/reclamos.csv", "reclamos.csv")

reclamos = pd.read_csv("reclamos.csv")   # muestra de 20.000 reclamos SERNAC 2011 (dato histórico)
print(reclamos.shape)
reclamos[["producto", "mercado"]].head()

## 1. Clases con muy pocos ejemplos

Aquí el «texto» es corto: el **producto** reclamado (`"LINEA DE CREDITO Y CUENTA CORRIENTE"`), y la etiqueta es el **mercado** (financieros, telecomunicaciones…). Antes de entrenar, mira cuántos reclamos tiene cada mercado: uno tiene **un solo** reclamo, y con un solo ejemplo `stratify` no puede repartirlo entre entrenamiento y prueba (lanza `ValueError`). Decisión declarada: trabajamos con los mercados que tienen **al menos 50 reclamos**.

### ✍️ Ejercicio 1 — Filtra los mercados con suficientes ejemplos

In [ ]:
conteo = reclamos["mercado"].value_counts()
mercados_ok = conteo[conteo >= 50].index
rec = reclamos[reclamos["mercado"].isin(mercados_ok)]
n_mercados = rec["mercado"].nunique()
print("mercados:", reclamos["mercado"].nunique(), "→", n_mercados, "| reclamos:", len(reclamos), "→", len(rec))
print(conteo.tail(6))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert rec["mercado"].value_counts().min() >= 50
    assert n_mercados == int((conteo >= 50).sum())
    assert len(rec) == int(conteo[conteo >= 50].sum())
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'mercados_ok = conteo[conteo >= 50].index; rec = reclamos[reclamos["mercado"].isin(mercados_ok)]; n_mercados = rec["mercado"].nunique().')
    print("   Detalle:", e)

## 2. El mismo pipeline, con texto corto y muchos datos

Repite la receta de la lección: split estratificado (25 % prueba, `random_state=42`), TF-IDF ajustado solo con entrenamiento (aquí basta el `TfidfVectorizer()` por defecto: el texto ya viene en mayúsculas sin tildes), regresión logística, exactitud y línea base.

### ✍️ Ejercicio 2 — Entrena y evalúa producto → mercado

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    rec["producto"], rec["mercado"], test_size=0.25, stratify=rec["mercado"], random_state=42)
vec = TfidfVectorizer()
modelo = LogisticRegression(max_iter=1000).fit(vec.fit_transform(X_tr), y_tr)
pred = modelo.predict(vec.transform(X_te))
exactitud = accuracy_score(y_te, pred)
linea_base = y_te.value_counts().max() / len(y_te)
print(f"exactitud: {exactitud:.1%} | línea base: {linea_base:.1%} ({y_te.value_counts().idxmax()})")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert abs(exactitud - accuracy_score(y_te, pred)) < 1e-12
    assert exactitud > 0.9 and linea_base < 0.4
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'modelo = LogisticRegression(max_iter=1000).fit(vec.fit_transform(X_tr), y_tr); pred = modelo.predict(vec.transform(X_te)).')
    print("   Detalle:", e)

## 3. La exactitud esconde a las minorías

Una exactitud de ~95 % parece un éxito. Pero los mercados grandes (financieros, telecomunicaciones) dominan el promedio. El **recall** de un mercado responde: de los reclamos que *de verdad* eran de ese mercado, ¿qué fracción detectó el modelo? Encuentra el peor.

### ✍️ Ejercicio 3 — Encuentra el mercado peor detectado

In [ ]:
reporte = classification_report(y_te, pred, output_dict=True, zero_division=0)
recall = {m: reporte[m]["recall"] for m in sorted(set(y_te))}
peor = min(recall, key=recall.get)
print(f"peor mercado: {peor} (recall {recall[peor]:.0%}, {int((y_te == peor).sum())} reclamos en prueba)")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(recall) == n_mercados
    assert peor == min(recall, key=recall.get)
    assert recall[peor] < 0.5 < exactitud
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'recall = {m: reporte[m]["recall"] for m in sorted(set(y_te))}; peor = min(recall, key=recall.get).')
    print("   Detalle:", e)

## 4. ¿Por qué falla? El texto no trae la información

Mira los productos del peor mercado: *ELECTRÓNICA, COMPUTACIÓN, MUEBLES…* — los mismos que se reclaman en **locales comerciales**. El canal (tienda física o internet) no está en el texto del producto, así que **ningún** modelo de palabras puede distinguirlos. Mide qué fracción de los reclamos del peor mercado tiene un producto que también aparece en locales comerciales.

### ✍️ Ejercicio 4 — Mide el solapamiento de productos

In [ ]:
prod_peor = set(rec.loc[rec["mercado"] == peor, "producto"])
prod_locales = set(rec.loc[rec["mercado"] == "LOCALES COMERCIALES", "producto"])
compartidos = prod_peor & prod_locales
fraccion = rec.loc[rec["mercado"] == peor, "producto"].isin(compartidos).mean()
print(f"{len(compartidos)} productos compartidos | {fraccion:.0%} de los reclamos de {peor} usan uno de ellos")

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert compartidos == prod_peor & prod_locales
    assert abs(fraccion - rec.loc[rec["mercado"] == peor, "producto"].isin(compartidos).mean()) < 1e-12
    assert fraccion > 0.5
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'compartidos = prod_peor & prod_locales; fraccion = rec.loc[rec["mercado"] == peor, "producto"].isin(compartidos).mean().')
    print("   Detalle:", e)

## Para llevar

- **Pocos ejemplos** por clase rompen la estratificación: agrupa, filtra o consigue más datos — y **declara** la decisión.
- Una exactitud global alta puede convivir con clases que el modelo casi nunca detecta: revisa el **recall por clase**.
- Si el texto no contiene la información (el canal de venta), ningún modelo la inventa. Es un problema de **datos**, no de algoritmo.

---

*Fin de la profundización de R3-01.*